# **Natural Language Processing - Assignment 1**
## **Mission Logs: Rebuilding Mary's Ears**

In [ ]:
import json, re, math, nltk
for r in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
'vader_lexicon']:
  nltk.download(r, quiet=True)
with open('mission_data.json') as f: # upload your file to Colab first
  data = json.load(f)
print('NLTK version:', nltk.__version__)
print(list(data['basic'].keys()))
print(list(data['advanced'].keys()))

NLTK version: 3.9.1
['paragraph', 'words', 'docs', 'target', 'roles', 'vectors']
['log', 'train', 'test']


In [ ]:
para = data["basic"]["paragraph"]
print(para)

Mary can't read the message; she says the words are broken. Dr. Grace says the engines aren't responding, and fuel is running low! Nothing was normal about that night, but nobody was panicking.


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

para = data["basic"]["paragraph"]

b1_sentences = len(sent_tokenize(para))
b1_split_tokens = len(para.split())
b1_nltk_tokens = len(word_tokenize(para))

print("Number of sentences:", b1_sentences)
print("Tokens using split():", b1_split_tokens)
print("Tokens using NLTK:", b1_nltk_tokens)

Number of sentences: 3
Tokens using split(): 33
Tokens using NLTK: 41


In [ ]:
from nltk.corpus import stopwords

para_lower = para.lower()
tokens = word_tokenize(para_lower)

stop_words = set(stopwords.words("english"))

# Keep not, no, and nor
stop_words -= {"not", "no", "nor"}

b2_clean_tokens = [
    token for token in tokens
    if token.isalpha() and token not in stop_words
]

print(b2_clean_tokens)

['mary', 'ca', 'read', 'message', 'says', 'words', 'broken', 'grace', 'says', 'engines', 'responding', 'fuel', 'running', 'low', 'nothing', 'normal', 'night', 'nobody', 'panicking']


In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

b3_table = {}

for word, pos in data["basic"]["words"]:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos)
    b3_table[word] = [stem, lemma]

# Find the pair that gets the same stem but has different lemmas
b3_overstemmed_pair = None

words = list(b3_table.keys())

for i in range(len(words)):
    for j in range(i + 1, len(words)):
        w1, w2 = words[i], words[j]

        if b3_table[w1][0] == b3_table[w2][0] and b3_table[w1][1] != b3_table[w2][1]:
            b3_overstemmed_pair = [w1, w2]
            break

    if b3_overstemmed_pair:
        break

print(b3_table)
print("Overstemmed pair:", b3_overstemmed_pair)

{'worse': ['wors', 'bad'], 'flies': ['fli', 'fly'], 'better': ['better', 'good'], 'caring': ['care', 'care'], 'university': ['univers', 'university'], 'universe': ['univers', 'universe']}
Overstemmed pair: ['university', 'universe']


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = data["basic"]["docs"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

b4_vocab_size = len(vectorizer.vocabulary_)
b4_row_sums = X.sum(axis=1).A1.tolist()

target_word, doc_index = data["basic"]["target"]

# Manual TF-IDF calculation
doc_tokens = docs[doc_index].split()
tf = doc_tokens.count(target_word) / len(doc_tokens)

df = sum(target_word in doc.split() for doc in docs)
N = len(docs)

idf = math.log10(N / df)
b4_tfidf = round(tf * idf, 3)

print("Vocabulary size:", b4_vocab_size)
print("Row sums:", b4_row_sums)
print("TF-IDF:", b4_tfidf)

Vocabulary size: 7
Row sums: [5, 5, 5, 5]
TF-IDF: 0.025


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np

roles = data["basic"]["roles"]

label_encoder = LabelEncoder()
b5_label_encoded = label_encoder.fit_transform(roles).tolist()

one_hot_encoder = OneHotEncoder(sparse_output=False)
one_hot = one_hot_encoder.fit_transform(
    np.array(roles).reshape(-1, 1)
)

b5_onehot_shape = list(one_hot.shape)

print("Label encoded:", b5_label_encoded)
print("One-hot shape:", b5_onehot_shape)

Label encoded: [3, 2, 1, 2, 0, 1, 1, 3]
One-hot shape: [8, 4]


In [ ]:
vectors = data["basic"]["vectors"]

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))

    magnitude_a = math.sqrt(sum(x * x for x in a))
    magnitude_b = math.sqrt(sum(y * y for y in b))

    return dot_product / (magnitude_a * magnitude_b)


words = list(vectors.keys())

b6_cosines = {}

for i in range(len(words)):
    for j in range(i + 1, len(words)):
        w1 = words[i]
        w2 = words[j]

        similarity = cosine_similarity(vectors[w1], vectors[w2])
        b6_cosines[f"{w1}-{w2}"] = round(similarity, 3)

b6_most_similar = max(b6_cosines, key=b6_cosines.get)

print("Cosine similarities:", b6_cosines)
print("Most similar pair:", b6_most_similar)

Cosine similarities: {'planet-crew': 0.895, 'planet-probe': 0.6, 'crew-probe': 0.43}
Most similar pair: planet-crew


In [ ]:
answers_basic = {
    "b1_sentences": b1_sentences,
    "b1_split_tokens": b1_split_tokens,
    "b1_nltk_tokens": b1_nltk_tokens,

    "b2_clean_tokens": b2_clean_tokens,

    "b3_table": b3_table,
    "b3_overstemmed_pair": b3_overstemmed_pair,

    "b4_vocab_size": b4_vocab_size,
    "b4_row_sums": b4_row_sums,
    "b4_tfidf": b4_tfidf,

    "b5_label_encoded": b5_label_encoded,
    "b5_onehot_shape": b5_onehot_shape,

    "b6_cosines": b6_cosines,
    "b6_most_similar": b6_most_similar
}

In [ ]:
import json

print(json.dumps(answers_basic))

{"b1_sentences": 3, "b1_split_tokens": 33, "b1_nltk_tokens": 41, "b2_clean_tokens": ["mary", "ca", "read", "message", "says", "words", "broken", "grace", "says", "engines", "responding", "fuel", "running", "low", "nothing", "normal", "night", "nobody", "panicking"], "b3_table": {"worse": ["wors", "bad"], "flies": ["fli", "fly"], "better": ["better", "good"], "caring": ["care", "care"], "university": ["univers", "university"], "universe": ["univers", "universe"]}, "b3_overstemmed_pair": ["university", "universe"], "b4_vocab_size": 7, "b4_row_sums": [5, 5, 5, 5], "b4_tfidf": 0.025, "b5_label_encoded": [3, 2, 1, 2, 0, 1, 1, 3], "b5_onehot_shape": [8, 4], "b6_cosines": {"planet-crew": 0.895, "planet-probe": 0.6, "crew-probe": 0.43}, "b6_most_similar": "planet-crew"}


## **MISSION LOG 2: Decoding the Galaxy**

In [ ]:
log = data["advanced"]["log"]

patterns = {
    "stardate": r"\bSD\d{4}\.\d{2}\b",
    "signal": r"\bSIG-[A-Z]{3}-\d{4}\b",
    "callsign": r"(?<![\w.-])@[a-z_]+\d*\b",
    "freq": r"\b\d+\.\d{2}\sGHz\b",
    "email": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
}

a1_stardate = {
    "count": len(re.findall(patterns["stardate"], log)),
    "third": re.findall(patterns["stardate"], log)[2]
}

a1_signal = {
    "count": len(re.findall(patterns["signal"], log)),
    "third": re.findall(patterns["signal"], log)[2]
}

a1_callsign = {
    "count": len(re.findall(patterns["callsign"], log)),
    "third": re.findall(patterns["callsign"], log)[2]
}

a1_freq = {
    "count": len(re.findall(patterns["freq"], log)),
    "third": re.findall(patterns["freq"], log)[2]
}

a1_email = {
    "count": len(re.findall(patterns["email"], log)),
    "third": re.findall(patterns["email"], log)[2]
}

print(a1_stardate)
print(a1_signal)
print(a1_callsign)
print(a1_freq)
print(a1_email)

{'count': 10, 'third': 'SD4687.60'}
{'count': 10, 'third': 'SIG-VUN-8255'}
{'count': 10, 'third': '@rocky89'}
{'count': 10, 'third': '16.58 GHz'}
{'count': 10, 'third': 'ryland.g@hailmary.space'}


In [ ]:
def clean_log(text):
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\b[\w.-]+@[\w.-]+\.\w+\b", " ", text)
    text = re.sub(r"@[a-z_]+\d*\b", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned = clean_log(log)

words = cleaned.split()

a2_word_count = len(words)
a2_longest_word = max(words, key=lambda word: (len(word), word))

print("Word count:", a2_word_count)
print("Longest word:", a2_longest_word)

Word count: 207
Longest word: understand


In [ ]:
greedy_matches = re.findall(r"<msg>.*</msg>", log, re.DOTALL)

lazy_matches = re.findall(r"<msg>(.*?)</msg>", log, re.DOTALL)

a3_msg_count = len(lazy_matches)
a3_longest_msg = max(lazy_matches, key=lambda message: (len(message), message))

print("Greedy count:", len(greedy_matches))
print("Message count:", a3_msg_count)
print("Longest message:", a3_longest_msg)

Greedy count: 1
Message count: 14
Longest message: The antennas were catching faint radio waves from the stars.


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, FreqDist

stop_words = set(stopwords.words("english"))
stop_words -= {"not", "no", "nor"}

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    elif tag.startswith("V"):
        return "v"
    elif tag.startswith("N"):
        return "n"
    elif tag.startswith("R"):
        return "r"
    return "n"

all_lemmas = []

for message in lazy_matches:
    message = message.lower()
    message = re.sub(r"[^a-z\s]", " ", message)

    tokens = word_tokenize(message)

    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    tagged = pos_tag(tokens)

    message_lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
    ]

    all_lemmas.extend(message_lemmas)

freq = FreqDist(all_lemmas)

a4_top5 = freq.most_common(5)

print(a4_top5)

[('repair', 3), ('ship', 2), ('computer', 2), ('damage', 2), ('memory', 2)]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from nltk.sentiment import SentimentIntensityAnalyzer

train_data = data["advanced"]["train"]
test_data = data["advanced"]["test"]

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    tagged = pos_tag(tokens)

    lemmas = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
    ]

    return " ".join(lemmas)

train_texts = [preprocess(text) for text, label in train_data]
train_labels = [label for text, label in train_data]

test_texts = [preprocess(text) for text in test_data]

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

classifier = MultinomialNB()
classifier.fit(X_train, train_labels)

a5_model_pred = classifier.predict(X_test).tolist()

sia = SentimentIntensityAnalyzer()

a5_vader = []

for text in test_data:
    compound = sia.polarity_scores(text)["compound"]

    if compound >= 0.05:
        sentiment = 1
    elif compound <= -0.05:
        sentiment = 0
    else:
        sentiment = None

    a5_vader.append(sentiment)

print("Model predictions:", a5_model_pred)
print("VADER:", a5_vader)

Model predictions: [1, 0, 1]
VADER: [1, 0, 1]


In [ ]:
answers_advanced = {
    "a1_stardate": a1_stardate,
    "a1_signal": a1_signal,
    "a1_callsign": a1_callsign,
    "a1_freq": a1_freq,
    "a1_email": a1_email,

    "a2_word_count": a2_word_count,
    "a2_longest_word": a2_longest_word,

    "a3_msg_count": a3_msg_count,
    "a3_longest_msg": a3_longest_msg,

    "a4_top5": a4_top5,

    "a5_model_pred": a5_model_pred,
    "a5_vader": a5_vader
}

import json
print(json.dumps(answers_advanced))

{"a1_stardate": {"count": 10, "third": "SD4687.60"}, "a1_signal": {"count": 10, "third": "SIG-VUN-8255"}, "a1_callsign": {"count": 10, "third": "@rocky89"}, "a1_freq": {"count": 10, "third": "16.58 GHz"}, "a1_email": {"count": 10, "third": "ryland.g@hailmary.space"}, "a2_word_count": 207, "a2_longest_word": "understand", "a3_msg_count": 14, "a3_longest_msg": "The antennas were catching faint radio waves from the stars.", "a4_top5": [["repair", 3], ["ship", 2], ["computer", 2], ["damage", 2], ["memory", 2]], "a5_model_pred": [1, 0, 1], "a5_vader": [1, 0, 1]}


## **Captain's Debrief**

**Prediction Log**

Before running the classifier ,I expected the TF-IDF + Multinomial Naive Bayes model to classify the text msgs resonably well because the training examples contained clear diffs between the labels . I also expected Vader to give diff results for messages with clearly positive or negative wording.

**Biggest Bug**

My biggest bug was assuming that the training data was stored as dictionaries with "text" and "label" keys. It was actually stored as pairs of text and labels, which caused a TypeError. I fixed this by unpacking each item as text, label.

**Help Declaration**

I used ChatGPT to understand the NLP concepts, debug errors, and clarify the assignment requirements. I also used it to help implement and correct parts of the code.